# EDA — Exploración inicial del dataset VIVSO

**PP2 · ITSE** — Programa de viviendas sociales, Subsecretaría de Promoción Humana, Santiago del Estero

Responde tres preguntas: ¿cuántos registros hay?, ¿qué tan completos son?, ¿cómo se distribuyen las variables clave?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
DRIVE = Path('/content/drive/MyDrive/PP2')
Path('img').mkdir(exist_ok=True)

df = pd.read_csv(DRIVE / 'viviendas_sinteticas.csv')

# La API Java devuelve columnas en camelCase (avanceObra, tipoVivienda);
# el CSV local generado por synthetic/generate.py usa snake_case.
# Detectamos el nombre en tiempo de ejecución para que el notebook funcione con ambos orígenes.
col_avance = 'avance_obra' if 'avance_obra' in df.columns else 'avanceObra'
col_tipo   = 'tipo_vivienda' if 'tipo_vivienda' in df.columns else 'tipoVivienda'
col_dorm   = 'cant_dormitorios' if 'cant_dormitorios' in df.columns else 'cantDormitorios'
col_dias   = 'dias_activa'
print(f'Registros: {len(df)} | Columnas: {df.shape[1]}')
df.head(3)

In [ ]:
info = pd.DataFrame({'tipo': df.dtypes, 'nulos': df.isnull().sum(),
                     'nulos_%': (df.isnull().sum()/len(df)*100).round(1),
                     'unicos': df.nunique()})
info[info['nulos'] > 0].sort_values('nulos_%', ascending=False)

### Interpretación de los nulos

- **`cluster` al 100%:** se calcula en el notebook 04; no existe en este CSV todavía.
- **`fecha_fin`:** obra sin terminar no tiene fecha de fin — no es dato faltante, es coherente con el estado de la obra.
- **`cuit_org`:** viviendas sin ONG gestora asignada (~20%) — relevante para el ministerio porque implica obras sin responsable externo.
- **`barrio` y `observacion`:** campos optativos por definición del sistema.

In [ ]:
# Paleta estándar del proyecto (se reutiliza en todos los notebooks)
C = {'base':'#4f46e5','ok':'#10b981','medio':'#f59e0b','alerta':'#f43f5e','neutro':'#94a3b8'}
COL_E = {'Iniciada':C['medio'],'Avanzada':C['base'],'Finalizada':C['ok'],'Adjudicada':'#818cf8'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Estados de obra — el indicador operativo principal del programa
c_e = df['estado'].value_counts()
axes[0].bar(c_e.index, c_e.values, color=[COL_E.get(e, C['neutro']) for e in c_e.index])
axes[0].bar_label(axes[0].containers[0], fmt='%d', padding=3)
axes[0].set_title('Viviendas por estado')

# Lente de priorización: de los 15 códigos, solo dos gatillan acción directa —
# 2b (riesgo de derrumbe, urgencia) y 3a (discapacidad, prioridad social).
# El resto se agrupa para no perder el contexto del padrón sin saturar el gráfico.
n_2b   = (df['clasificacion']=='2b').sum()
n_3a   = (df['clasificacion']=='3a').sum()
n_incl = (df['criterio']=='Inclusion').sum() - n_2b - n_3a
n_otro = (df['criterio']!='Inclusion').sum()

cats = ['Otros criterios\n(Otro / Exclusión)', 'Resto admitidas\n(1a, 2a)',
        '2b · Riesgo derrumbe\n(urgencia)', '3a · Discapacidad\n(prioridad social)']
vals = [n_otro, n_incl, n_2b, n_3a]
cols = [C['neutro'], C['ok'], C['alerta'], C['base']]
axes[1].barh(cats, vals, color=cols, alpha=0.9)
axes[1].bar_label(axes[1].containers[0], fmt='%d', padding=3)
axes[1].set_title('Padrón con lente de priorización')

plt.tight_layout(); plt.savefig('img/01_estados_prioridad.png', bbox_inches='tight'); plt.show()
print(f'Casos prioritarios (2b + 3a): {n_2b + n_3a} de {len(df)} ({(n_2b+n_3a)/len(df)*100:.1f}%)')

In [ ]:
# Verificación técnica — ¿qué proporción de obras activas fue visitada?
# La visita técnica es el único mecanismo de validación independiente del reporte de la ONG.
# Sin visita, el avance registrado es solo lo que declaró la organización — sin contraste en terreno.
df_vis = pd.read_csv(DRIVE / 'visitas.csv')

activas_ids = df[df['estado'].isin(['Iniciada','Avanzada'])]['num_exp']
visitadas   = set(df_vis['vivienda_id'].unique())
n_sin_visita = (~activas_ids.isin(visitadas)).sum()
pct_sin      = n_sin_visita / len(activas_ids) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ¿Cuántas obras activas nunca recibieron una visita?
axes[0].bar(['Con al menos\n1 visita', 'Sin ninguna\nvisita'],
            [len(activas_ids)-n_sin_visita, n_sin_visita],
            color=['#22c55e', '#ef4444'], alpha=0.85, width=0.5)
for bar, val in zip(axes[0].patches, [len(activas_ids)-n_sin_visita, n_sin_visita]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f'{val}\n({val/len(activas_ids)*100:.0f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title(f'Obras activas sin verificacion tecnica: {pct_sin:.0f}%')
axes[0].set_ylabel('Obras activas')

# Distribución de visitas por obra (solo las que tuvieron al menos una)
vis_por_obra = df_vis.groupby('vivienda_id').size().value_counts().sort_index()
axes[1].bar(vis_por_obra.index.astype(str), vis_por_obra.values, color='#2563eb', alpha=0.85)
axes[1].bar_label(axes[1].containers[0], fmt='%d', padding=3)
axes[1].set_title('Distribución de visitas por obra (obras visitadas)')
axes[1].set_xlabel('Visitas realizadas')

plt.tight_layout(); plt.savefig('img/01_visitas.png', bbox_inches='tight'); plt.show()
print(f'Obras activas       : {len(activas_ids)}')
print(f'Sin ninguna visita  : {n_sin_visita} ({pct_sin:.1f}%)')
print(f'Con al menos 1 visita: {len(activas_ids)-n_sin_visita} ({100-pct_sin:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histograma del AFO: revela si las obras están concentradas en etapas tempranas o hay progresión uniforme
axes[0].hist(df[col_avance].dropna(), bins=20, color='#2563eb', edgecolor='white', alpha=0.85)
axes[0].axvline(df[col_avance].mean(), color='#ef4444', linestyle='--',
                label=f'Media: {df[col_avance].mean():.1f}%')
axes[0].set_title('Distribución del avance (AFO)')
axes[0].set_xlabel('Avance (%)')
axes[0].legend(fontsize=9)

tc = df[col_tipo].value_counts()
axes[1].pie(tc.values, labels=tc.index, autopct='%1.1f%%',
            colors=['#3b82f6','#22c55e','#f59e0b'], startangle=90)
axes[1].set_title('Tipo de vivienda')

dc = df[col_dorm].value_counts().sort_index()
axes[2].bar(dc.index.astype(str), dc.values, color='#6366f1', alpha=0.85)
axes[2].set_title('Cantidad de dormitorios')

plt.tight_layout(); plt.savefig('img/01_avance_tipo.png', bbox_inches='tight'); plt.show()
print(df[col_avance].describe().round(1))

In [ ]:
# Distribución geográfica y obras en riesgo
PLAZO = 90   # plazo contractual de la etapa de construcción (días)
if col_dias not in df.columns:
    df['fecha_inic_dt'] = pd.to_datetime(df['fecha_inic'], format='%d-%m-%Y', errors='coerce')
    df['fecha_fin_dt']  = pd.to_datetime(df['fecha_fin'],  format='%d-%m-%Y', errors='coerce')
    df[col_dias] = (df['fecha_fin_dt'].fillna(pd.Timestamp.today()) - df['fecha_inic_dt']).dt.days

# Obra en riesgo de demora = activa, ya superó el plazo de 90 días y todavía no está
# por terminar (avance < 80%). Si el dataset ya trae nivel_riesgo (lo calcula el ETL),
# se usa directo; si no, se reconstruye la regla acá.
if 'nivel_riesgo' in df.columns:
    en_riesgo = df['nivel_riesgo'].isin(['alto','medio'])
else:
    en_riesgo = df['estado'].isin(['Iniciada','Avanzada']) & (df[col_dias] > PLAZO) & (df[col_avance] < 80)
n = en_riesgo.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
por_depto = df['departamento'].value_counts().sort_values()
axes[0].barh(por_depto.index, por_depto.values, color='#2563eb', alpha=0.85)
axes[0].bar_label(axes[0].containers[0], fmt='%d', padding=3, fontsize=8)
axes[0].set_title('Viviendas por departamento')

axes[1].bar(['Sin riesgo','En riesgo'], [len(df)-n, n], color=['#22c55e','#ef4444'], alpha=0.85)
for bar, val in zip(axes[1].patches, [len(df)-n, n]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val), ha='center', fontsize=11)
axes[1].set_title(f'Obras en riesgo de demora — {n} ({n/len(df)*100:.1f}%)')

plt.tight_layout(); plt.savefig('img/01_depto_riesgo.png', bbox_inches='tight'); plt.show()

In [ ]:
# Resumen ejecutivo
e = df['estado'].value_counts()
activas = e.get('Iniciada',0) + e.get('Avanzada',0)
term    = e.get('Finalizada',0) + e.get('Adjudicada',0)
print('='*50 + '\nRESUMEN EJECUTIVO\n' + '='*50)
print(f'Total viviendas         : {len(df)}')
print(f'Departamentos cubiertos : {df["departamento"].nunique()}')
print(f'Obras activas           : {activas} ({activas/len(df)*100:.1f}%)')
print(f'Obras terminadas        : {term} ({term/len(df)*100:.1f}%)')
print(f'Avance promedio (AFO)   : {df[col_avance].mean():.1f}%')
print(f'En riesgo de demora     : {n} ({n/len(df)*100:.1f}%)')
print(f'Sin verificacion tecnica: {n_sin_visita} de {activas} obras activas ({pct_sin:.1f}%)')